In [2]:
import pandas as pd
import ray
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
import plotly.graph_objects as go
from itertools import combinations
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import (train_test_split, GridSearchCV,
                                    StratifiedKFold, StratifiedShuffleSplit,
                                    cross_val_score)
from sklearn.metrics import make_scorer, recall_score, precision_score, f1_score, classification_report
from imblearn.ensemble import BalancedRandomForestClassifier
import numpy as np
import joblib
from pandarallel import pandarallel
from sklearn.preprocessing import LabelEncoder, StandardScaler
import re

2025-05-26 14:49:05.721039: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-26 14:49:05.775452: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-26 14:49:06.019591: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-26 14:49:06.019652: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-26 14:49:06.021117: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to regi

In [3]:
def busqueda(expression,column,name=None):
    if column is np.nan:
        return np.nan
    m = re.search(expression,column)
    if m == None:
        return np.nan
    if name != None:
        return name
    else :
        return m.group(0)

In [92]:
df_2 = pd.read_csv("/home/nicolas/nico/Data/Masivas/data_correccion_zari/ZariDR3_2arcscSkiff.csv")
df_2 = df_2.reset_index()

In [93]:
lista = pd.DataFrame()

In [94]:
skiff = df_2[['index','source_id',"skiff_type",'Bibcode','GroupID_skiff']].astype(str)

In [95]:
skiff['skiff_type'] = skiff['skiff_type'].replace('nan', np.nan)
skiff['Bibcode'] = skiff['Bibcode'].replace('nan', np.nan)
skiff['GroupID_skiff'] = skiff['GroupID_skiff'].replace('nan', np.nan)

In [96]:
skiff = skiff.loc[skiff['skiff_type'].notna()]
print(f"{len(skiff)} Estrellas no unicas con informacion espectral en skiff")

123133 Estrellas no unicas con informacion espectral en skiff


In [97]:
lista["sp_skiff"] = [len(skiff)]

In [98]:
resultado = skiff.apply(lambda row: busqueda(r":|\?", row["skiff_type"]), axis=1)
sp_peculiaridades = skiff.loc[resultado.notna()]["skiff_type"].unique()
print(f"Estrellas con uncertanity in their sp \n {sp_peculiaridades}")
skiff = skiff.loc[resultado.isna()]
lista["uncertanity"] = [len(resultado.loc[resultado.notna()])]

Estrellas con uncertanity in their sp 
 ['A3:' 'O9III:' 'B1:e' ... 'O9.5I?p' 'B3IV:n' 'O7.5IV((f)) + O9III:']


In [99]:
resultado  = skiff.apply(lambda row: busqueda("\+",row["skiff_type"]),axis=1)
print(f"{len(resultado.loc[resultado.notna()])} Distarted Binaries stars")
skiff = skiff.loc[resultado.isna()].reset_index(drop=True)
lista["binaries"] = [len(resultado.loc[resultado.notna()])]

In [460]:
resultado  = skiff.apply(lambda row: busqueda("em",row["skiff_type"]),axis=1)
sp_em = skiff.loc[resultado.notna()]["skiff_type"].unique()
print(f"{len(resultado.loc[resultado.notna()])} stars with em \n {sp_em}")
skiff = skiff.loc[resultado.isna()].reset_index(drop=True)
lista["em"] = [len(resultado.loc[resultado.notna()])]

5885 stars with em 
 ['em' 'O em' 'em pec' 'em e']


In [461]:
resultado = skiff.apply(lambda row: busqueda(r"OBe", row["skiff_type"]), axis=1)
sp_OBe = skiff.loc[resultado.notna()]["skiff_type"].unique()
print(f"Estrellas de tipo OBe \n {sp_OBe}")
skiff = skiff.loc[resultado.isna()]
lista["sp_OBe"] = [len(resultado.loc[resultado.notna()])]

Estrellas de tipo OBe 
 ['OBe']


In [462]:
resultado = skiff.apply(lambda row: busqueda(r"OB", row["skiff_type"]), axis=1)
sp_OB = skiff.loc[resultado.notna()]["skiff_type"].unique()
print(f"Estrellas de tipo OB \n {sp_OB}")
uniques_OB  = skiff.loc[resultado.notna()].drop_duplicates(subset="source_id")
skiff = skiff.loc[resultado.isna()]
lista["sp_OB"] = [len(resultado.loc[resultado.notna()])]

Estrellas de tipo OB 
 ['OB' 'OB-' 'OB-e' 'OB pec' 'OB-n' 'OB-k' 'OBnn' 'OB-(e)' 'OBn' 'OB-nn'
 'OB- pec']


In [464]:
resultado = skiff.apply(lambda row: busqueda(r"W", row["skiff_type"]), axis=1)
sp_WR = skiff.loc[resultado.notna()]["skiff_type"].unique()
print(f"Estrellas de tipo WR \n {sp_WR}")
skiff = skiff.loc[resultado.isna()]
lista["WR"] = [len(resultado.loc[resultado.notna()])]

Estrellas de tipo WR 
 ['WN6o' 'WN5.5-A' 'WR pec' 'WN6h' 'WN6' 'WN/Of' 'WN' 'WN5' 'WN7' 'WN7-A'
 'WN8h' 'WN3ha' 'WR' 'WN3' 'WN4-A' 'WN6-A' 'WN6ha' 'WN pec' 'WNE' 'WN7ha'
 'WC7' 'O6f/WR' 'WRe' 'WN9' 'WN9ha' 'WC' 'WN8' 'WN8(h)' 'WN4'
 '[WC]-PG1159' '[WR]' 'WN5/6' 'WN6.5-A' 'WN6-B' '[WC5/6]' '[WC4/6]' 'WN7o'
 'WN7/8' 'WN4o' 'WN6-A(B)' 'WN6-s' 'WN7b' 'O2If*/WN6' 'O2If*/WN5' 'WC5'
 'WCE/WN' 'WC7e' 'WN8-A' 'Of/WN7' 'Ofpe/WN9' 'WN3/5' 'WN6 (weak)' 'WN11h'
 'WN6(h)' 'O3If*/WN6-A' 'WN3o pec' 'WN9/10' 'WN9h' '[WC11]' 'WN7o/WC'
 'WN5o' 'WN4/5o' 'WN8/O6.5If' 'O3.5If*/WN7' 'WN4.5' 'WN5-A(B)' 'WN6/7'
 'WC4' 'WN5sh' 'WO1' 'WO' 'WC5 pec' 'WC4 pec' 'O2.5If*/WN6' 'WN4h/WCE'
 'WC8.5' 'WC9' 'WCe' 'WN7(h)' 'WN7h' 'Of/[WR]' 'WC7/WN6' 'WN8o/WC7'
 '[WC6/7]' '[WC8]' '[WC7]']


In [465]:
resultado = skiff.apply(lambda row: busqueda(r"Be", row["skiff_type"]), axis=1)
sp_Be = skiff.loc[resultado.notna()]["skiff_type"].unique()
print(f"Estrellas de tipo Be \n {sp_Be}")
skiff = skiff.loc[resultado.isna()]
lista["Be"] = [len(resultado.loc[resultado.notna()])]

Estrellas de tipo Be 
 ['Be' 'Bep' 'Be shell' 'Ae/Be' 'Bek' 'Beq' 'Be shell pec']


In [466]:
resultado = skiff.apply(lambda row: busqueda(r"sd", row["skiff_type"]), axis=1)
sp_sd = skiff.loc[resultado.notna()]["skiff_type"].unique()
print(f"Estrellas de tipo sd \n {sp_sd}")
skiff = skiff.loc[resultado.isna()]
lista["sd"] = [len(resultado.loc[resultado.notna()])]

Estrellas de tipo sd 
 ['sdB' 'sdOA' 'sdBO' 'sd' 'sdB8III He2' 'sdB3I He8' 'sdB2II He15'
 'sdB3IV He35' 'sdBk' 'sdA2' 'sdB7III He3' 'sdB5III He7' 'sdB3IV He12'
 'sdB0.5VIp He9' 'sdB7III He2' 'sdB8III He0' 'sdB3IV He5' 'sdB8IV He2'
 'sdO' 'sdB0.5Vp He7' 'sdB3' 'sdBC0II He40' 'sdB5III He11' 'sdOC9I He40'
 'sdOC9.5II-III He40' 'sdA' 'sdBC' 'sdB2.5IV He11' 'sdB2VIIp He4'
 'sdB2.5II He24']


In [467]:
resultado = skiff.apply(lambda row: busqueda(r"(e|em|H|w|wl|wk|v|k|ak|Sr|Si|sh|shell|h|p|pec|s|n|Cr|nn|He|m|Fe|Ca|Mg|Na|Ti|CN|:|\?)", row["skiff_type"]), axis=1)
sp_peculiaridades = skiff.loc[resultado.notna()]["skiff_type"].unique()
print(f"Estrellas con peculiaridades \n {sp_peculiaridades}")
skiff = skiff.loc[resultado.isna()]
lista["pec"] = [len(resultado.loc[resultado.notna()])]

Estrellas con peculiaridades 
 ['ApSi' 'B2nk' 'B1.5Vnep He-n' ... 'kA8hF1mF5' 'B6/7IIIn' 'B3Iae']


In [468]:
resultado = skiff.apply(lambda row: busqueda(r"(N|C|PN|R|DA|S|Q|L)", row["skiff_type"]), axis=1)
sp_NC = skiff.loc[resultado.notna()]["skiff_type"].unique()
print(f"Estrellas con N strong or C strong or PN \n {sp_NC}")
skiff = skiff.loc[resultado.isna()]
lista["PN_or_C_or_N"]= [len(resultado.loc[resultado.notna()])]

Estrellas con N strong or C strong or PN 
 ['ON9.7Iab' 'ON9Ia' 'ON9.2Iab' 'DA' 'ON9.5III' 'ON6V((f))' 'BN2.5III'
 'OC7III/V' 'PN' 'BC1.5Iab' 'ON6.5V' 'OC9V' 'ON9.7Ib' 'ON9.7I' 'R' 'ON8V'
 'OC9.5I' 'OC9.7Ia' 'OC9.2Ia' 'R0' 'C' 'OC9.5Iab' 'OC9Iab' 'OC9.7Iab'
 'B2V CII' 'R2' 'C1,2' 'C0' 'C2,2' 'BN0IV' 'OC7.5III((f))' 'S' 'BC0Ia'
 'BN0Ia' 'OC5/6Vz' 'ON2III(f*)' 'ON3III(f*)' 'R1' 'BN0.5II-III' 'M5S'
 'BC1Ia' 'LBV' 'ON7V((f))' 'ON5.5V((f))' 'BC1.5Ia' 'M3.5S' 'BN0.5Ia'
 'BN1Ia' 'BC2Ia' 'ON9.7Ia' 'OC9.7Ib' 'OC9.5II' 'BC2Iab' 'BN1Ib' 'BQ[]'
 'G8/R0' 'K3/R3' 'ON7III(f)' 'ON7.5' 'BN2Ib' 'C4,1' 'R3' 'ON6V((f))z'
 'ON9.5V' 'ON9.2V' 'ON9V' 'ON7V' 'ON9.2IV' 'ON9.5IV' 'ON8.5V' 'BC2Ib']


In [469]:
lista["Non-MK"] = lista["pec"]+lista["PN_or_C_or_N"]+lista["WR"]+lista["sd"]

In [470]:
lista

,sp_skiff,uncertanity,binaries,em,sp_OBe,WR,Be,sd,pec,PN_or_C_or_N,Non-MK,sp_OB
0,123133,8810,2110,5885,591,309,1025,76,7761,206,8352,15332


## Deje de hacer la seleccion de tipos espectrales y ahora estoy corrigiendo unos tipos espectrales 

In [109]:
resultado = skiff.apply(lambda row: busqueda(r"^[A-Z/]+$", row["skiff_type"]), axis=1)

In [85]:

# Function to generate the new type
def generate_type(type_str):
    types = type_str.split('/')
    new_types = []
    
    for t in types:
        primary_type = t
        subtype_value = np.random.randint(0,9)
        new_types.append(f"{primary_type}{subtype_value}")
    
    return '/'.join(new_types)

In [86]:
skiff['mk'] = skiff.loc[resultado.notna()]["skiff_type"].apply(generate_type)

In [87]:
resultado = skiff.apply(lambda row: busqueda(r"^[OBAFKGM]\d(?:\.\d)?/\d", row["skiff_type"]), axis=1)

In [88]:
def spectral_to_number(spectral_type):
    letter = spectral_type[0]
    number = spectral_type[1:].replace('/', '.')
    base_number = replace_map[letter]
    return f"{base_number}.{number}"

In [89]:
skiff.loc[resultado.notna(),"mk"] = resultado

In [90]:
sp = skiff.loc[resultado.notna()]["mk"].str.split("/").str[0].str[0]

In [91]:
subtipe_1 = skiff.loc[resultado.notna()]["mk"].str.split("/").str[0].str[1]

In [92]:
subtipe_2 = skiff.loc[resultado.notna()]["mk"].str.split("/").str[1].str[0]

In [28]:
skiff.loc[resultado.notna(),"mk"] =  sp + subtipe_1+"/"+sp+subtipe_2

In [29]:
resultado = skiff.apply(lambda row: busqueda(r'[OBAFKGM]\d(?:\.\d)?/[OBAFKGM]\d(?:\.\d)?', row["skiff_type"]), axis=1)

In [30]:
skiff.loc[resultado.notna(),"mk"] = resultado

In [31]:
resultado = skiff.apply(lambda row: busqueda(r'[OBAFGKM]/[A-Za-z]\d$|[A-Za-z]\d/[OBAFGKM]$', row["skiff_type"]), axis=1)

In [32]:
def convert_to_number(value):
    match = re.match(r'([OBAFGKM])(\d?)/([OBAFGKM])(\d?)$', value)
    if match:
        letter1, num1, letter2, num2 = match.groups()
        
        num1 = num1 if num1 else str(np.random.randint(0, 9))
        num2 = num2 if num2 else str(np.random.randint(0, 9))
        
        return f"{letter1}{num1}/{letter2}{num2}"
    return None

values = ['O/B0', 'O/B2', 'F8/G']
converted_values = [convert_to_number(value) for value in values]

In [33]:
skiff.loc[resultado.notna(), 'mk'] = skiff.loc[resultado.notna(), 'skiff_type'].apply(convert_to_number)


In [34]:
resultado = skiff.apply(lambda row: busqueda(r'[OBAFGKM]\d', row["skiff_type"]), axis=1)

In [35]:
skiff.loc[(resultado.notna())&(skiff["mk"].isna()),"mk"] = resultado.loc[skiff["mk"].isna()]

In [36]:
skiff.loc[skiff["mk"].isna(), "mk"] = (
    skiff.loc[skiff["mk"].isna(), "skiff_type"].str[0] + 
    np.random.randint(0, 9, skiff["mk"].isna().sum()).astype(str)
)

In [38]:
skiff['mk'] = skiff['mk'].str.replace('.', '')

In [39]:
replace_map = {
    'O': '0.',
    'B': '1.',
    'A': '2.',
    'F': '3.',
    'G': '4.',
    'K': '5.',
    'M': '6.'
}

skiff['mk'] = skiff['mk'].replace(replace_map, regex=True)

In [40]:
def promedio_redondeado(valor):
    if '/' in valor:
        numeros = [float(n) for n in valor.split('/')]
        return np.random.uniform(numeros[0],numeros[1])
    else:
        return valor


        
# Aplica la función a la columna 'mk'
skiff['mk'] = skiff['mk'].apply(promedio_redondeado).astype(float)

In [41]:
coded = "mk"

In [42]:
import pandas as pd

def process_skiff_dataframe(skiff, std_threshold,coded):
    # Filtra los datos en función de la columna "GroupID_skiff"
    stars_unicas = skiff.loc[skiff["GroupID_skiff"].isna()].reset_index(drop=True)
    star_no_unicas = skiff.loc[skiff["GroupID_skiff"].notna()].reset_index(drop=True)

    # Agrupa por "GroupID_skiff" y obtiene los tipos únicos de "skiff_type"
    star_no_unicas = star_no_unicas.merge(
        star_no_unicas.groupby("GroupID_skiff").agg({
            "Bibcode": lambda x: list(x.unique()),
            "index": lambda x: list(x.unique()),
            "skiff_type": lambda x: list(x.unique()),
            "source_id": lambda x: list(x.unique())
        }).reset_index(),
        on="GroupID_skiff",
        suffixes=('', '_list')
    )

    # Calcula la desviación estándar por grupo y filtra los datos según el umbral
    std_by_group = star_no_unicas.groupby("GroupID_skiff")[coded].transform("std")
    star_no_unicas["std_sp"] = std_by_group
    star_no_unicas = star_no_unicas.loc[(star_no_unicas["std_sp"] < std_threshold) | (star_no_unicas["std_sp"].isna())]

    # Calcula la media por grupo y la asigna como etiqueta
    mean_by_group = star_no_unicas.groupby("GroupID_skiff")[coded].transform("mean")
    star_no_unicas["label"] = mean_by_group

    # Elimina duplicados
    star_no_unicas = star_no_unicas.drop_duplicates(subset="GroupID_skiff")

    # Asigna etiquetas y lista de tipos para estrellas únicas
    stars_unicas["label"] = stars_unicas[coded]
    stars_unicas["skiff_type_list"] = stars_unicas["skiff_type"]

    # Combina ambos DataFrames
    skiff_mk = pd.concat([star_no_unicas, stars_unicas])

    return skiff_mk


In [43]:
skiff_mk = process_skiff_dataframe(skiff, 0.3,"mk")

In [45]:
skiff_mk["label"] = np.round(skiff_mk["label"],1)

In [47]:
skiff_mk.to_csv("/home/nicolas/nico/Data/Masivas/Data_OB_stars/skiff_2arcsec_OBAGDR3_coded_V2.csv",index=False)

In [63]:
skiff_mk["source_id"] = skiff_mk["source_id"].astype(int)
skiff_mk["index"] = skiff_mk["index"].astype(int)
skiff_mk["GroupID_skiff"] = skiff_mk["GroupID_skiff"].astype(float)

In [68]:
df_2 = df_2.merge(skiff_mk[['index', 'source_id', 'skiff_type', 'Bibcode', 'GroupID_skiff', 'mk',
       'Bibcode_list', 'index_list', 'skiff_type_list', 'source_id_list',
       'std_sp', 'label']], on=['source_id','index','skiff_type','GroupID_skiff','Bibcode'], how='left')

In [ ]:
df_2.to_csv("/home/nicolas/nico/Data/Masivas/Data_OB_stars/skiff_2arcsec_OBAGDR3_prep_V2.csv",
           index=False)

In [472]:
df_3 = pd.read_csv("/home/nicolas/nico/Data/Masivas/Data_OB_stars/skiff_2arcsec_OBAGDR3_prep_V2.csv")

In [475]:
estrellas_con_al_menos_otra_class = len(uniques_OB.loc[uniques_OB["GroupID_skiff"].notna()])

In [477]:
estrellas_con_al_menos_otra_class/ len(uniques_OB)

0.7189761215629522

In [360]:
OB_not_coded = df_2.loc[(df_2["GroupID_skiff"].notna())&(df_2["skiff_type"]=="OB")]

In [361]:
coded = df_3.loc[(df_3["mk"].notna())&(df_3["GroupID_skiff"].notna())]

In [362]:
coded = coded[["mk","GroupID_skiff"]].merge(OB_not_coded,how="inner",on="GroupID_skiff")

In [450]:
uniques_OB_coded = len(coded.drop_duplicates(subset="source_id"))

In [452]:
(uniques_OB_coded/uniques_OB)*100

42.94500723589002